# RidgeVisionNet -- Part 3 of 3 (v2): Robustness + Statistical Validation + Compute Cost

Fresh re-run using the corrected `full` model (EfficientNetB0 rescaling fix
applied). Note: Section 5's statistical validation (texture features vs.
blood-group label) does not depend on the CNN at all, so its results should
be numerically identical to the original run -- included here only for a
single self-contained final results folder.

**Before running:** attach your fingerprint dataset + `ridgevisionnet-v2-part2-output`.


In [1]:
# !pip install -q scikit-image
import os, gc, json, time
from pathlib import Path

import cv2
import numpy as np
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from scipy import stats
from scipy.optimize import minimize_scalar

print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))


TensorFlow: 2.20.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [2]:
# =========================
# OFFLINE / NO-INTERNET RESILIENCE FOR IMAGENET WEIGHTS
# =========================
# Kaggle sessions default to "Internet: Off". Every backbone in this notebook
# (RidgeVisionNet + 6 of the 10 baselines) loads ImageNet-pretrained weights,
# which needs to download a .h5 file from storage.googleapis.com the first
# time it's used. If you saw an error like:
#   URLError: <urlopen error [Errno -3] Temporary failure in name resolution>
# it means Internet is Off for this session.
#
# FIX (do this first, it's the real fix): right sidebar -> Settings ->
# Internet -> toggle ON -> Save, then re-run. This changes nothing about the
# methodology -- it only lets the same published ImageNet weights download.
#
# The helpers below are a safety net for cases where you can't turn Internet
# on (e.g. a no-internet competition): they (1) detect the problem early with
# a clear message instead of a deep stack trace, (2) reuse any already-
# downloaded weights from a locally attached Kaggle dataset if one is
# present, and (3) as a last resort let backbone-building fall back to
# random-initialized weights (with a loud warning) so the notebook can still
# run to completion rather than crash -- note this last case means that
# backbone is no longer using transfer learning, which should be reported as
# a deviation if it happens for your real (non-QUICK_RUN) results.

import socket
import shutil


def internet_available(host="storage.googleapis.com", port=443, timeout=3):
    try:
        socket.getaddrinfo(host, port)
        return True
    except OSError:
        return False


HAS_INTERNET = internet_available()
print("Internet reachable:", HAS_INTERNET)


def stage_offline_imagenet_weights():
    """If a Kaggle dataset containing pre-downloaded Keras ImageNet weight
    files is attached (search Kaggle Datasets for 'keras pretrained models'
    or similar), copy any .h5 files found under /kaggle/input into
    ~/.keras/models/ so tf.keras.applications finds them in its local cache
    and skips the network call entirely. Safe to call even if nothing is
    found (returns 0)."""
    cache_dir = Path.home() / ".keras" / "models"
    cache_dir.mkdir(parents=True, exist_ok=True)
    found = 0
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for h5_path in input_root.rglob("*.h5"):
            target = cache_dir / h5_path.name
            if not target.exists():
                shutil.copy(h5_path, target)
                found += 1
    if found:
        print(f"Staged {found} local weight file(s) into {cache_dir}.")
    else:
        print("No local ImageNet weight files found under /kaggle/input.")
    return found


if not HAS_INTERNET:
    staged = stage_offline_imagenet_weights()
    if not staged:
        print()
        print("ACTION SUGGESTED: no internet reachable AND no local weights dataset found.")
        print("Go to Settings -> Internet -> ON (right sidebar), Save, and re-run this cell.")
        print("If your competition disallows internet, attach a Kaggle dataset that hosts the")
        print("*_notop.h5 files for the backbones used here, then re-run this cell.")


def load_backbone(backbone_cls, weights="imagenet", **kwargs):
    """Build a tf.keras.applications backbone, falling back to random-init
    weights (with a loud, impossible-to-miss warning) if the requested
    weights can't be obtained. Keeps the notebook runnable end-to-end even
    when Internet is Off and no offline weights dataset is attached; does
    NOT fix the underlying cause -- see the cell above."""
    try:
        return backbone_cls(weights=weights, **kwargs)
    except Exception as e:
        if weights is None:
            raise
        print(f"WARNING: could not load '{weights}' weights for {backbone_cls.__name__} ({type(e).__name__}: {e}).")
        print("Falling back to random initialization (weights=None) so the run can continue.")
        print("This backbone will NOT benefit from ImageNet transfer learning until the")
        print("internet/offline-weights issue above is resolved and this cell is re-run.")
        return backbone_cls(weights=None, **kwargs)


Internet reachable: True


In [3]:
# =========================
# CONFIG
# =========================
OUTPUT_DIR = Path('/kaggle/working')
RESULTS_DIR = OUTPUT_DIR / 'ridgevisionnet_results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CLASS_LABELS = ['A+', 'A-', 'AB+', 'AB-', 'B+', 'B-', 'O+', 'O-']
LABEL_TO_INDEX = {label: i for i, label in enumerate(CLASS_LABELS)}
NUM_CLASSES = len(CLASS_LABELS)
SEED = 42

# --- Speed settings, changed this pass to fit a single Kaggle GPU session ---
# Mixed precision (fp16 compute, fp32 master weights) is a standard, purely
# computational optimization on GPUs with tensor cores (T4/P100/V100/A100) --
# it does not change what the model learns, only how fast the same forward/
# backward math runs, typically ~1.5-2x on these GPUs.
tf.keras.mixed_precision.set_global_policy('mixed_float16')

BATCH_SIZE = 32  # was 16; larger batches use the GPU more efficiently
IMG_SIZE = 224  # RidgeVisionNet / most baselines; InceptionV3 baseline overrides to 299

QUICK_RUN = False  # set False for the real run used in the paper
# EPOCHS_HEAD/EPOCHS_FINE are now upper bounds, not targets -- EarlyStopping
# (added this pass, see train_model) will stop well before these ceilings for
# any model that has converged or plateaued, which was most of the wasted
# time in the previous run (e.g. mobilenet_v2 ran all 55 epochs stuck near
# chance accuracy). Lowering the ceiling itself additionally caps worst-case
# runtime for a model that never triggers early stopping.
EPOCHS_HEAD = 3 if QUICK_RUN else 8
EPOCHS_FINE = 5 if QUICK_RUN else 30  # NOT lowered further than this: your own log showed resnet50/densenet121 flat until fine-tune epoch ~14, then climbing through epoch 40 -- a lower ceiling would cut those off mid-breakthrough and understate their real accuracy. EarlyStopping (patience=4) does the actual time-saving for models that plateau, not this ceiling.
# N_FOLDS reduced from 5 to 3: 3-fold stratified CV is still a standard,
# citable, defensible choice (commonly used exactly for compute-constrained
# settings) -- report it as "3-fold" rather than "5-fold" in the paper's
# Experimental Setup section rather than silently changing the number.
N_FOLDS = 2 if QUICK_RUN else 3
N_PERMUTATIONS = 200 if QUICK_RUN else 2000
MC_DROPOUT_SAMPLES = 10 if QUICK_RUN else 30

np.random.seed(SEED)
tf.random.set_seed(SEED)
print('QUICK_RUN =', QUICK_RUN)
print('Mixed precision policy:', tf.keras.mixed_precision.global_policy())


QUICK_RUN = False
Mixed precision policy: <DTypePolicy "mixed_float16">


In [4]:
# =========================
# RESTORE PRIOR-STAGE OUTPUTS
# =========================
# Attach the Kaggle Dataset from Part 2's output as Input before running.
import shutil
from pathlib import Path

_restored = 0
for _src in Path('/kaggle/input').rglob('ridgevisionnet_results'):
    if _src.is_dir():
        for _f in _src.rglob('*'):
            if _f.is_file():
                _target = RESULTS_DIR / _f.relative_to(_src)
                _target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy(_f, _target)
                _restored += 1
print(f'Restored {_restored} file(s) into {RESULTS_DIR} from attached prior-stage dataset(s).')
if _restored == 0:
    print('WARNING: nothing restored -- did you attach the previous part\'s output dataset as Input?')


Restored 5 file(s) into /kaggle/working/ridgevisionnet_results from attached prior-stage dataset(s).


In [5]:
# =========================
# DATASET AUTO-DETECTION (same convention as the v1 notebook)
# =========================

def find_dataset_dir():
    search_roots = [Path('/kaggle/input'), Path('/kaggle/working')]
    candidates = []
    for root in search_roots:
        if not root.exists():
            continue
        for path in root.rglob('*'):
            if not path.is_dir():
                continue
            class_folder_count = sum((path / label).is_dir() for label in CLASS_LABELS)
            if class_folder_count >= 6:
                candidates.append((class_folder_count, path))
    if not candidates:
        raise FileNotFoundError(
            'Dataset not found. Attach the fingerprint blood group dataset to Kaggle, '
            'or set DATASET_DIR manually below.'
        )
    candidates.sort(key=lambda item: item[0], reverse=True)
    print('Auto-detected dataset folder:', candidates[0][1])
    return candidates[0][1]

DATASET_DIR = find_dataset_dir()

all_paths, all_labels = [], []
for label in CLASS_LABELS:
    files = sorted((DATASET_DIR / label).glob('*'))
    all_paths.extend(files)
    all_labels.extend([LABEL_TO_INDEX[label]] * len(files))
all_labels = np.array(all_labels)
print(f'Total images: {len(all_paths)}')
for label in CLASS_LABELS:
    print(f'  {label}: {(all_labels == LABEL_TO_INDEX[label]).sum()}')


Auto-detected dataset folder: /kaggle/input/datasets/sravani2006/fingerprint-blood-group-classification-dataset/datasets
Total images: 5837
  A+: 402
  A-: 1009
  AB+: 708
  AB-: 761
  B+: 652
  B-: 741
  O+: 852
  O-: 712


In [6]:
# =========================
# IMAGE LOADING
# =========================

def load_rgb(path, img_size):
    img = cv2.imread(str(path))
    if img is None:
        return np.zeros((img_size, img_size, 3), dtype=np.float32)
    img = cv2.resize(img, (img_size, img_size))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img.astype(np.float32) / 255.0


class FingerprintSequence(tf.keras.utils.Sequence):
    """Simple image-only generator (RidgeVisionNet computes its own ridge
    orientation field on-device from the image, so no separate texture-vector
    input is needed here, unlike the v1 dual-input generator)."""

    def __init__(self, paths, labels, img_size, batch_size=BATCH_SIZE, augment=False, shuffle=True, **kwargs):
        super().__init__(**kwargs)  # required by Keras 3's PyDataset base (tf.keras.utils.Sequence is now an alias for it)
        self.paths = paths
        self.labels = labels
        self.img_size = img_size
        self.batch_size = batch_size
        self.augment = augment
        self.shuffle = shuffle
        self.indices = np.arange(len(paths))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.paths) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def _augment(self, img):
        if np.random.rand() < 0.5:
            angle = np.random.uniform(-10, 10)
            m = cv2.getRotationMatrix2D((self.img_size / 2, self.img_size / 2), angle, 1.0)
            img = cv2.warpAffine(img, m, (self.img_size, self.img_size), borderMode=cv2.BORDER_REFLECT)
        if np.random.rand() < 0.5:
            img = np.clip(img * np.random.uniform(0.85, 1.15) + np.random.uniform(-0.05, 0.05), 0, 1)
        if np.random.rand() < 0.3:
            img = np.clip(img + np.random.normal(0, 0.02, img.shape), 0, 1)
        return img.astype(np.float32)

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        images = np.zeros((len(batch_idx), self.img_size, self.img_size, 3), dtype=np.float32)
        labels = np.zeros(len(batch_idx), dtype=np.int32)
        for i, bi in enumerate(batch_idx):
            img = load_rgb(self.paths[bi], self.img_size)
            if self.augment:
                img = self._augment(img)
            images[i] = img
            labels[i] = self.labels[bi]
        return images, labels


## RidgeVisionNet components

Ridge orientation field (deterministic), ROAM (orientation-gated attention),
Adaptive Gated Fusion, and the full model builder -- identical logic to
`backend/ml/models/{ridge_orientation,roam,ridgevision_net}.py` in the
repository, inlined here so this notebook is self-contained on Kaggle.

In [7]:
# =========================
# Ridge Orientation Field (deterministic, no trainable params)
# =========================
class RidgeOrientationField(tf.keras.layers.Layer):
    def __init__(self, block_size=8, **kwargs):
        super().__init__(**kwargs)
        self.block_size = block_size
        self.sobel_x = tf.constant([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=tf.float32)[:, :, None, None]
        self.sobel_y = tf.constant([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=tf.float32)[:, :, None, None]

    def call(self, inputs):
        gx = tf.nn.conv2d(inputs, self.sobel_x, strides=1, padding='SAME')
        gy = tf.nn.conv2d(inputs, self.sobel_y, strides=1, padding='SAME')
        gxx, gyy, gxy = gx * gx, gy * gy, gx * gy
        pool = lambda t: tf.nn.avg_pool2d(t, ksize=self.block_size, strides=self.block_size, padding='VALID')
        vxx, vyy, vxy = pool(gxx), pool(gyy), pool(gxy)
        numerator = 2.0 * vxy
        denominator = vxx - vyy
        theta2 = tf.atan2(numerator, denominator)
        cos2, sin2 = tf.cos(theta2), tf.sin(theta2)
        energy = tf.sqrt(numerator**2 + denominator**2)
        coherence = tf.clip_by_value(energy / (vxx + vyy + 1e-6), 0.0, 1.0)
        return tf.concat([cos2, sin2, coherence], axis=-1)

    def get_config(self):
        config = super().get_config(); config.update({'block_size': self.block_size}); return config


class ROAM(tf.keras.layers.Layer):
    """Ridge Orientation Attention Module."""
    def __init__(self, reduction=4, use_channel_gate=True, **kwargs):
        super().__init__(**kwargs)
        self.reduction = reduction
        # FIX: use_channel_gate must be a real constructor arg that build()/call()
        # branch on. Previously callers tried to monkeypatch `roam.channel_excite`
        # with a Lambda *before* the first call -- but build() runs on that first
        # call and unconditionally overwrote it with a trainable Dense layer, so
        # the "no_roam_channel_gate" ablation silently trained an unmodified model.
        self.use_channel_gate = use_channel_gate

    def build(self, input_shapes):
        feature_shape, _ = input_shapes
        channels = int(feature_shape[-1])
        reduced = max(channels // self.reduction, 8)
        self.orientation_proj = tf.keras.layers.Conv2D(reduced, 3, padding='same', activation='relu')
        self.spatial_gate = tf.keras.layers.Conv2D(1, 1, padding='same', activation='sigmoid')
        if self.use_channel_gate:
            self.channel_squeeze = tf.keras.layers.Dense(reduced, activation='relu')
            self.channel_excite = tf.keras.layers.Dense(channels, activation='sigmoid')
        self.resize_target = (int(feature_shape[1]), int(feature_shape[2]))

    def call(self, inputs):
        features, orientation_field = inputs
        orientation_resized = tf.image.resize(orientation_field, self.resize_target, method='bilinear')
        spatial_attention = self.spatial_gate(self.orientation_proj(orientation_resized))
        if self.use_channel_gate:
            channel_stats = tf.reduce_mean(features, axis=[1, 2])
            channel_attention = self.channel_excite(self.channel_squeeze(channel_stats))[:, None, None, :]
        else:
            channel_attention = 1.0  # true no-op: skips the gate entirely instead of shape-mismatched ones_like
        attended = features * spatial_attention * channel_attention
        return attended, spatial_attention

    def get_config(self):
        config = super().get_config()
        config.update({'reduction': self.reduction, 'use_channel_gate': self.use_channel_gate})
        return config


class AdaptiveGatedFusion(tf.keras.layers.Layer):
    def build(self, input_shapes):
        a_shape, b_shape = input_shapes
        dim = max(int(a_shape[-1]), int(b_shape[-1]))
        self.proj_a = tf.keras.layers.Dense(dim)
        self.proj_b = tf.keras.layers.Dense(dim)
        self.gate_dense = tf.keras.layers.Dense(dim, activation='sigmoid')

    def call(self, inputs):
        branch_a, branch_b = inputs
        a, b = self.proj_a(branch_a), self.proj_b(branch_b)
        gate = self.gate_dense(tf.concat([a, b], axis=-1))
        return gate * a + (1.0 - gate) * b, gate


In [8]:
# =========================
# RidgeVisionNet builder (+ ablation-variant builder)
# =========================

def build_ridgevision_net(img_size=IMG_SIZE, num_classes=NUM_CLASSES, dropout_rate=0.35,
                           trainable_backbone_layers=40, use_orientation_field=True,
                           use_channel_gate=True, fusion_mode='adaptive',
                           use_ridge_branch=True, use_appearance_branch=True, name='ridgevision_net'):
    assert use_ridge_branch or use_appearance_branch
    image_input = tf.keras.Input(shape=(img_size, img_size, 3), name='fingerprint_image')
    # FIX: tf.keras.applications.EfficientNetB0 has a built-in Rescaling(1/255)
    # layer expecting raw [0,255] pixels. Our shared pipeline already scales
    # images to [0,1] (Section 7.3), so without this correction EfficientNetB0
    # was dividing already-scaled pixels by 255 again -- crushing its input by
    # another factor of 255 and very likely causing the training collapses
    # observed in efficientnet_b0_plain, single_branch_texture, and
    # no_orientation_field. This undoes that scaling immediately before the
    # backbone; the grayscale/orientation-field path below is untouched since
    # it operates on the original [0,1] image_input, not this rescaled copy.
    efficientnet_input = tf.keras.layers.Rescaling(255.0, name='undo_pipeline_rescale_for_efficientnet')(image_input)
    backbone = load_backbone(tf.keras.applications.EfficientNetB0, weights='imagenet', include_top=False, input_tensor=efficientnet_input)
    for layer in backbone.layers:
        layer.trainable = False
    if trainable_backbone_layers > 0:
        for layer in backbone.layers[-trainable_backbone_layers:]:
            if not isinstance(layer, tf.keras.layers.BatchNormalization):
                layer.trainable = True

    branches = []
    if use_ridge_branch:
        mid_features = backbone.get_layer('block6a_expand_activation').output
        if use_orientation_field:
            # Keras 3: raw tf.* ops cannot be applied directly to a KerasTensor
            # outside of a Layer; wrap in Lambda so it becomes a proper graph op.
            grayscale = tf.keras.layers.Lambda(
                lambda x: tf.image.rgb_to_grayscale(x), name='to_grayscale'
            )(image_input)
            orientation_field = RidgeOrientationField(block_size=8, dtype='float32')(grayscale)  # force float32: mixed precision would otherwise feed float16 into the hardcoded float32 Sobel kernels
        else:
            orientation_field = tf.keras.layers.Lambda(
                lambda x: tf.ones((tf.shape(x)[0], tf.shape(x)[1], tf.shape(x)[2], 3)),
                name='dummy_orientation_field'
            )(mid_features)
        roam = ROAM(use_channel_gate=use_channel_gate)
        attended_mid, spatial_attention = roam([mid_features, orientation_field])
        ridge_branch = tf.keras.layers.GlobalAveragePooling2D()(attended_mid)
        branches.append(ridge_branch)
    else:
        spatial_attention = None

    if use_appearance_branch:
        appearance_branch = tf.keras.layers.GlobalAveragePooling2D()(backbone.output)
        branches.append(appearance_branch)

    if len(branches) == 1:
        fused = branches[0]
    elif fusion_mode == 'adaptive':
        fused, _ = AdaptiveGatedFusion()(branches)
    elif fusion_mode == 'static_average':
        dim = 256
        projected = [tf.keras.layers.Dense(dim)(b) for b in branches]
        fused = tf.keras.layers.Average()(projected)
    elif fusion_mode == 'concat':
        fused = tf.keras.layers.Concatenate()(branches)
    else:
        raise ValueError(fusion_mode)

    hidden = tf.keras.layers.Dense(256, activation='relu')(fused)
    hidden = tf.keras.layers.Dropout(dropout_rate, name='mc_dropout')(hidden)
    logits = tf.keras.layers.Dense(num_classes, name='logits', dtype='float32')(hidden)
    probs = tf.keras.layers.Softmax(name='blood_group', dtype='float32')(logits)

    outputs = {'blood_group': probs, 'logits': logits}
    if spatial_attention is not None:
        outputs['attention_map'] = spatial_attention
    return tf.keras.Model(inputs=image_input, outputs=outputs, name=name)


ABLATION_GRID = [
    dict(name='full'),
    dict(name='no_orientation_field', use_orientation_field=False),
    dict(name='no_roam_channel_gate', use_channel_gate=False),
    dict(name='static_fusion', fusion_mode='static_average'),
    dict(name='concat_fusion', fusion_mode='concat'),
    dict(name='single_branch_texture', use_ridge_branch=False),
    dict(name='single_branch_ridge', use_appearance_branch=False),
    dict(name='no_finetune', trainable_backbone_layers=0),
]
print(f'{len(ABLATION_GRID)} ablation variants configured.')


8 ablation variants configured.


In [9]:
# FIX: this array conversion originally lived inside Part 1's baseline-loop
# cell (not included here), but later cells need it.
all_paths_arr = np.array(all_paths, dtype=object)


In [10]:
# =========================
# RECREATE THE SAME TRAIN/VAL/TEST SPLIT USED IN PART 2 (v2)'S ABLATION STUDY
# =========================
train_idx, test_idx = train_test_split(np.arange(len(all_paths_arr)), test_size=0.15, stratify=all_labels, random_state=SEED)
train_idx, val_idx = train_test_split(train_idx, test_size=0.1765, stratify=all_labels[train_idx], random_state=SEED)

train_paths, train_labels = all_paths_arr[train_idx], all_labels[train_idx]
val_paths, val_labels = all_paths_arr[val_idx], all_labels[val_idx]
test_paths, test_labels = all_paths_arr[test_idx], all_labels[test_idx]

val_seq = FingerprintSequence(val_paths, val_labels, IMG_SIZE, augment=False, shuffle=False)
test_seq = FingerprintSequence(test_paths, test_labels, IMG_SIZE, augment=False, shuffle=False)

FULL_MODEL_WEIGHTS_PATH = RESULTS_DIR / 'ridgevision_full_model.weights.h5'
assert FULL_MODEL_WEIGHTS_PATH.exists(), (
    f'{FULL_MODEL_WEIGHTS_PATH} not found -- did Part 2 (v2)\'s output dataset attach correctly?'
)
print('Recreated split. Full model weights found at', FULL_MODEL_WEIGHTS_PATH)


Recreated split. Full model weights found at /kaggle/working/ridgevisionnet_results/ridgevision_full_model.weights.h5


In [11]:
# Rebuild the 'full' RidgeVisionNet architecture fresh and load the weights
# saved during the ablation loop above, rather than relying on a live Python
# reference held across many other model builds (that pattern is exactly
# what caused the OOM/kernel-death crash previously -- see the ablation cell).
full_model = build_ridgevision_net(name='full_reloaded_for_calibration')
full_model.load_weights(str(FULL_MODEL_WEIGHTS_PATH))
print('Reloaded full model weights from', FULL_MODEL_WEIGHTS_PATH)


I0000 00:00:1783904616.355576      22 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1783904616.358689      22 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Reloaded full model weights from /kaggle/working/ridgevisionnet_results/ridgevision_full_model.weights.h5


## 4. Robustness under perturbation (Table 7.3 / Section 7.3)

Evaluates the trained `full` model's accuracy under blur, noise, occlusion,
rotation, and downsampling, at 3 severities each, using only the test split.

In [12]:
def perturb(img, kind, severity):
    img = img.copy()
    if kind == 'blur':
        k = [3, 7, 11][severity]
        return cv2.GaussianBlur(img, (k, k), 0)
    if kind == 'noise':
        sigma = [0.01, 0.03, 0.06][severity]
        return np.clip(img + np.random.normal(0, sigma, img.shape), 0, 1).astype(np.float32)
    if kind == 'occlusion':
        frac = [0.1, 0.25, 0.4][severity]
        h, w = img.shape[:2]
        ch, cw = int(h * frac), int(w * frac)
        y0, x0 = np.random.randint(0, h - ch + 1), np.random.randint(0, w - cw + 1)
        img[y0:y0 + ch, x0:x0 + cw] = 0
        return img
    if kind == 'rotation':
        angle = [5, 10, 15][severity]
        m = cv2.getRotationMatrix2D((img.shape[1] / 2, img.shape[0] / 2), angle, 1.0)
        return cv2.warpAffine(img, m, (img.shape[1], img.shape[0]), borderMode=cv2.BORDER_REFLECT)
    if kind == 'downsample':
        factor = [2, 4, 8][severity]
        h, w = img.shape[:2]
        small = cv2.resize(img, (w // factor, h // factor))
        return cv2.resize(small, (w, h))
    raise ValueError(kind)


robustness_results = {}
for kind in ['blur', 'noise', 'occlusion', 'rotation', 'downsample']:
    robustness_results[kind] = []
    for severity in range(3):
        images = np.zeros((len(test_paths), IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
        for i, p in enumerate(test_paths):
            img = load_rgb(p, IMG_SIZE)
            images[i] = perturb(img, kind, severity)
        probs = full_model.predict(images, verbose=0, batch_size=BATCH_SIZE)['blood_group']
        acc = accuracy_score(test_labels, probs.argmax(axis=1))
        robustness_results[kind].append({'severity': severity, 'accuracy': float(acc)})
        print(f'{kind} severity={severity}: acc={acc:.4f}')

with open(RESULTS_DIR / 'robustness_results.json', 'w') as f:
    json.dump(robustness_results, f, indent=2)


I0000 00:00:1783904662.616747      69 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


blur severity=0: acc=0.8984
blur severity=1: acc=0.8516
blur severity=2: acc=0.6667
noise severity=0: acc=0.9007
noise severity=1: acc=0.9007
noise severity=2: acc=0.8973
occlusion severity=0: acc=0.8767
occlusion severity=1: acc=0.8379
occlusion severity=2: acc=0.6849
rotation severity=0: acc=0.9041
rotation severity=1: acc=0.8847
rotation severity=2: acc=0.8151
downsample severity=0: acc=0.8950
downsample severity=1: acc=0.3961
downsample severity=2: acc=0.3459


## 5. Biological-plausibility statistical validation (Section 7.5, at full Kaggle scale)

Same method as `analysis/statistical_validation.py` (run earlier on a
1,200-image CPU-feasible subsample); rerun here on the full Kaggle-hosted
corpus as a larger-scale corroboration. Read Section 5.1 before interpreting
these numbers.

In [13]:
from skimage.feature import local_binary_pattern, graycomatrix, graycoprops
from skimage.measure import shannon_entropy

def extract_stat_features(gray):
    lbp = local_binary_pattern(gray, P=8, R=1, method='uniform')
    hist, _ = np.histogram(lbp, bins=10, range=(0, 10), density=True)
    glcm = graycomatrix(gray, distances=[1], angles=[0], levels=256, symmetric=True, normed=True)
    return {
        'lbp_uniformity': float(hist.max()),
        'lbp_peak': float(np.argmax(hist)),
        'glcm_contrast': float(graycoprops(glcm, 'contrast')[0, 0]),
        'glcm_homogeneity': float(graycoprops(glcm, 'homogeneity')[0, 0]),
        'glcm_energy': float(graycoprops(glcm, 'energy')[0, 0]),
        'glcm_correlation': float(graycoprops(glcm, 'correlation')[0, 0]),
        'ridge_density': float(cv2.Canny(gray, 50, 150).mean() / 255.0),
        'intensity_entropy': float(shannon_entropy(gray)),
        'intensity_mean': float(gray.mean()),
        'intensity_std': float(gray.std()),
    }

def mutual_information_binned(x, y, bins=10):
    c_xy = np.histogram2d(x, y, bins=[bins, len(np.unique(y))])[0]
    p_xy = c_xy / c_xy.sum()
    p_x = p_xy.sum(axis=1, keepdims=True); p_y = p_xy.sum(axis=0, keepdims=True)
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = np.where(p_xy > 0, np.log(p_xy / (p_x * p_y + 1e-12) + 1e-12), 0.0)
    return float(np.sum(p_xy * ratio))

def permutation_test_mi(x, y, n_permutations, rng):
    observed = mutual_information_binned(x, y)
    permuted = np.empty(n_permutations)
    y_shuffled = y.copy()
    for i in range(n_permutations):
        rng.shuffle(y_shuffled)
        permuted[i] = mutual_information_binned(x, y_shuffled)
    return observed, float((np.sum(permuted >= observed) + 1) / (n_permutations + 1))

def eta_squared(groups):
    all_values = np.concatenate(groups)
    grand_mean = all_values.mean()
    ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
    ss_total = np.sum((all_values - grand_mean) ** 2)
    return float(ss_between / ss_total) if ss_total > 0 else 0.0

rng = np.random.default_rng(SEED)
rows, stat_labels = [], []
for p, lab in zip(test_paths, test_labels):  # test split only, consistent held-out set used throughout
    gray = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    gray = cv2.resize(gray, (256, 256)) if gray is not None else np.zeros((256, 256), np.uint8)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray = clahe.apply(gray)
    rows.append(extract_stat_features(gray))
    stat_labels.append(lab)
stat_labels = np.array(stat_labels)
feature_names = list(rows[0].keys())
feature_matrix = np.array([[r[n] for n in feature_names] for r in rows])

stat_results = {}
for j, fname in enumerate(feature_names):
    x = feature_matrix[:, j]
    groups = [x[stat_labels == c] for c in range(NUM_CLASSES)]
    f_stat, p_anova = stats.f_oneway(*groups)
    eta2 = eta_squared(groups)
    x_z = (x - x.mean()) / (x.std() + 1e-9)
    mi, perm_p = permutation_test_mi(x_z, stat_labels, N_PERMUTATIONS, rng)
    stat_results[fname] = {'anova_F': float(f_stat), 'anova_p': float(p_anova), 'eta_squared': eta2,
                            'mutual_information_bits': mi / np.log(2), 'permutation_p_value': perm_p}
    print(f"{fname:22s} F={f_stat:8.2f} p={p_anova:.2e} eta2={eta2:.4f} MI={mi/np.log(2):.4f} permp={perm_p:.4f}")

with open(RESULTS_DIR / 'statistical_validation_full_scale.json', 'w') as f:
    json.dump({'n_samples': len(stat_labels), 'features': stat_results}, f, indent=2)
print('\nSaved statistical_validation_full_scale.json -- compare eta-squared values against Table 3 in the paper.')


lbp_uniformity         F=   73.84 p=9.11e-84 eta2=0.3732 MI=0.3620 permp=0.0005
lbp_peak               F=   50.19 p=4.68e-60 eta2=0.2881 MI=0.2677 permp=0.0005
glcm_contrast          F=   29.25 p=2.33e-36 eta2=0.1909 MI=0.2001 permp=0.0005
glcm_homogeneity       F=   95.20 p=6.18e-103 eta2=0.4343 MI=0.4373 permp=0.0005
glcm_energy            F=  115.18 p=2.89e-119 eta2=0.4816 MI=0.4687 permp=0.0005
glcm_correlation       F=   35.68 p=6.12e-44 eta2=0.2235 MI=0.2387 permp=0.0005
ridge_density          F=   64.55 p=8.54e-75 eta2=0.3423 MI=0.3164 permp=0.0005
intensity_entropy      F=   96.01 p=1.27e-103 eta2=0.4364 MI=0.4429 permp=0.0005
intensity_mean         F=   49.25 p=4.77e-59 eta2=0.2843 MI=0.2817 permp=0.0005
intensity_std          F=   24.89 p=4.62e-31 eta2=0.1672 MI=0.1960 permp=0.0005

Saved statistical_validation_full_scale.json -- compare eta-squared values against Table 3 in the paper.


## 6. Computational cost (Table for compute-cost comparison)

In [14]:
import time as _time

def count_params(model):
    return int(np.sum([tf.size(v).numpy() for v in model.trainable_variables])), \
           int(np.sum([tf.size(v).numpy() for v in model.non_trainable_variables]))

def measure_latency(model, img_size, n_runs=30, is_dict_output=True):
    dummy = np.random.rand(1, img_size, img_size, 3).astype(np.float32)
    _ = model.predict(dummy, verbose=0)  # warmup
    start = _time.time()
    for _ in range(n_runs):
        _ = model.predict(dummy, verbose=0)
    return (_time.time() - start) / n_runs * 1000  # ms/image

trainable, non_trainable = count_params(full_model)
latency_ms = measure_latency(full_model, IMG_SIZE)
compute_cost = {
    'ridgevision_net': {
        'trainable_params': trainable,
        'non_trainable_params': non_trainable,
        'latency_ms_per_image_cpu_or_gpu_batch1': latency_ms,
    }
}
with open(RESULTS_DIR / 'compute_cost.json', 'w') as f:
    json.dump(compute_cost, f, indent=2)
print(json.dumps(compute_cost, indent=2))
print('\nRerun this cell for each baseline model to fill out the full compute-cost table in the paper.')


{
  "ridgevision_net": {
    "trainable_params": 8378217,
    "non_trainable_params": 2012071,
    "latency_ms_per_image_cpu_or_gpu_batch1": 82.00403054555257
  }
}

Rerun this cell for each baseline model to fill out the full compute-cost table in the paper.


## Save everything for pulling back into the paper

All JSON files under `ridgevisionnet_results/` in `/kaggle/working` map
directly onto: Table 1 (baseline_comparison_summary.json), Table 2
(ablation_results.json), Section 7.3 (robustness_results.json), Section 4.4 /
calibration (calibration_results.json), Section 7.5 corroboration
(statistical_validation_full_scale.json), and the compute-cost table
(compute_cost.json). Download the `ridgevisionnet_results/` folder from
Kaggle's output panel and use those numbers -- do not retype them from
memory or estimate them.